
# Modeling solidification in resilient mode
This example runs the `solidification workflow <Solidification_Simulation>`
with ``resilient_mode=True``.

Starting the transient solve from the steady solution with a time step of
0.1 s diverges, and Fluent raises a floating point exception. Without resilient
mode, the session is left in the diverged state. In resilient mode, a
checkpoint is written before ``calculate`` runs. When the solve fails, the
session is restored to that checkpoint and the original error is raised, so the
script can catch it, reduce the time step and solve again from the steady
solution.

See `ref_resilient_guide` for details.


## Perform required imports



In [ ]:
import ansys.fluent.core as pyfluent
from ansys.fluent.core import examples

## Launch Fluent in resilient mode



In [ ]:
solver_session = pyfluent.Solver.from_install(
    precision=pyfluent.Precision.DOUBLE,
    dimension=pyfluent.Dimension.TWO,
    resilient_mode=True,
)
resilient = solver_session.resilient
resilient.on_event(lambda event: print(f"[{event.kind.value}] {event.message}"))

## Read the mesh



In [ ]:
mesh_file = examples.download_file("solid.msh", "pyfluent/solidification")
solver_session.settings.file.read_mesh(file_name=mesh_file)

## Configure the solver



In [ ]:
# Built-in solver objects:
general = pyfluent.solver.General(settings_source=solver_session)
# Aliasing reused objects:
general_solver = general.solver

general_solver.two_dim_space.set_state("swirl")
general.operating_conditions.gravity.set_state({"enable": True, "components": [-9.81]})

## Enable the models
The energy model is enabled together with solidification and melting.



In [ ]:
viscous = pyfluent.solver.Viscous(settings_source=solver_session)
viscous.model.set_state("laminar")
solver_session.tui.define.models.solidification_melting(
    "yes", "constant", "100000", "yes", "no"
)

## Define the material



In [ ]:
materials = pyfluent.solver.Materials(settings_source=solver_session)
materials.database.copy_by_name(type="fluid", name="air", new_name="liquid-metal")
materials.fluid["liquid-metal"] = {
    "density": {
        "polynomial": {"coefficients": [8000, -0.1], "function_of": "temperature"},
        "option": "polynomial",
    },
    "viscosity": {"value": 0.00553},
    "specific_heat": {"value": 680.0},
    "thermal_conductivity": {"value": 30.0},
    "melting_heat": {"value": 100000.0},
    "tsolidus": {"value": 1150.0},
    "tliqidus": {"value": 1150.0},
}
fluid = pyfluent.solver.FluidCellZone(settings_source=solver_session, name="fluid")
fluid.general.material.set_state("liquid-metal")

## Set the boundary conditions



In [ ]:
inlet = pyfluent.solver.VelocityInlet(settings_source=solver_session, name="inlet")
inlet.momentum.velocity_magnitude.value.set_state(0.00101)
inlet.thermal.temperature.value.set_state(1300)

outlet = pyfluent.solver.VelocityInlet(settings_source=solver_session, name="outlet")
outlet_momentum = outlet.momentum
outlet_momentum.velocity_specification_method.set_state("Components")
outlet_momentum.swirl_angular_velocity.set_state(1)
outlet_momentum.velocity_components.set_state([0.001, 0, 0])
outlet.thermal.temperature.value.set_state(500)

bottom_wall = pyfluent.solver.WallBoundary(
    settings_source=solver_session, name="bottom-wall"
)
bottom_wall.thermal.set_state({"thermal_condition": "Temperature", "temperature": 1300})

free_surface = pyfluent.solver.WallBoundary(
    settings_source=solver_session, name="free-surface"
)
free_surface.momentum.set_state(
    {
        "shear_condition": "Marangoni Stress",
        "surface_tension_gradient": -0.00036,
    }
)
free_surface.thermal.set_state(
    {
        "thermal_condition": "Convection",
        "free_stream_temp": 1500,
        "heat_transfer_coeff": 100,
    }
)

side_wall = pyfluent.solver.WallBoundary(
    settings_source=solver_session, name="side-wall"
)
side_wall.thermal.set_state({"thermal_condition": "Temperature", "temperature": 1400})

solid_wall = pyfluent.solver.WallBoundary(
    settings_source=solver_session, name="solid-wall"
)
solid_wall.momentum.set_state(
    {
        "wall_motion": "Moving Wall",
        "velocity_spec": "Rotational",
        "rotation_speed": 1,
    }
)
solid_wall.thermal.set_state({"thermal_condition": "Temperature", "temperature": 500})

## Set the solution methods



In [ ]:
methods = pyfluent.solver.Methods(settings_source=solver_session)
methods.p_v_coupling.flow_scheme.set_state("Coupled")
methods.spatial_discretization.discretization_scheme.set_state({"pressure": "presto!"})
methods.pseudo_time_method.formulation.set_state({"coupled_solver": "global-time-step"})

## Solve the steady conduction problem



In [ ]:
controls = pyfluent.solver.Controls(settings_source=solver_session)
controls.equations.set_state({"flow": False, "w-swirl": False})

initialization = pyfluent.solver.Initialization(settings_source=solver_session)
initialization.hybrid_initialize()

results = pyfluent.solver.Results(settings_source=solver_session)
results.custom_field_functions.create(
    name="omegar", custom_field_function="1 * radial_coordinate"
)
calculate_patch = initialization.patch.calculate_patch
calculate_patch(
    cell_zones=["fluid"],
    variable="x-pull-velocity",
    reference_frame="Relative to Cell Zone",
    use_custom_field_function=True,
    custom_field_function_name="omegar",
    value=0.001,
)
calculate_patch(
    cell_zones=["fluid"],
    variable="z-pull-velocity",
    reference_frame="Relative to Cell Zone",
    use_custom_field_function=True,
    custom_field_function_name="omegar",
)

run_calculation = pyfluent.solver.RunCalculation(settings_source=solver_session)
run_calculation.pseudo_time_settings.time_step_method.set_state(
    {
        "time_step_method": "user-specified",
        "auto_time_size_calc_solid_zone": False,
    }
)
run_calculation.iterate(iter_count=20)

## Solve the transient problem and recover from the divergence
With a time step of 0.1 s the solve diverges. Resilient mode restores the
checkpoint written before ``calculate``, which holds the steady solution, and
raises the original error. The script then reduces the time step, increases
the under-relaxation and solves again from the steady solution.



In [ ]:
general_solver.time.set_state("unsteady-1st-order")
controls.equations.set_state({"flow": True, "w-swirl": True})
controls.under_relaxation.set_state({"delh": 0.1})

transient_controls = run_calculation.transient_controls
transient_controls.time_step_size.set_state(0.1)
transient_controls.time_step_count.set_state(50)
try:
    run_calculation.calculate()
except RuntimeError as error:
    print(f"Transient solve failed: {error}")
    print("Rolled back:", resilient.last_outcome.rolled_back)
    controls.under_relaxation.set_state({"delh": 0.02})
    transient_controls.time_step_size.set_state(0.001)
    transient_controls.time_step_count.set_state(5)
    run_calculation.calculate()

## Save and exit



In [ ]:
solver_session.settings.file.write_case_data(file_name="solidification_resilient")
solver_session.exit()